# Physical model validation

Use this notebook to validate the cycling physics model against both synthetic checks and real efforts with measured power.

The goal is to compare measured rider power against modeled resistive power and inspect the residuals:

`residual_w = measured_power_w - modeled_crank_power_w`

Positive residuals mean the model is underestimating the required rider power. Negative residuals mean the model is overestimating it.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "climbing_performance").exists() else NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

WindowsPath('c:/Users/Benjamin Lindblom/CPE')

In [2]:
import pandas as pd

from climbing_performance.gpx import parse_gpx
from climbing_performance.metrics import estimate_power_components
from climbing_performance.models import Bike, Climb, Rider
from climbing_performance.workflow import RouteSegmentAdjustment, summarise_gpx_performance

## 1. Synthetic sanity checks

These checks use simple inputs where the expected behavior is obvious. They are useful for catching equation or unit mistakes before using messy real-world GPX files.

In [3]:
def synthetic_case(
    name: str,
    *,
    rider_mass_kg: float = 70.0,
    bike_mass_kg: float = 8.0,
    cda_m2: float = 0.32,
    crr: float = 0.004,
    distance_m: float,
    elevation_gain_m: float,
    time_s: float,
    avg_altitude_m: float = 0.0,
    headwind_m_s: float = 0.0,
) -> dict:
    rider = Rider(mass_kg=rider_mass_kg)
    bike = Bike(
        mass_kg=bike_mass_kg,
        drag_coefficient=1.0,
        frontal_area_m2=cda_m2,
        rolling_resistance_coefficient=crr,
    )
    climb = Climb(
        distance_m=distance_m,
        elevation_gain_m=elevation_gain_m,
        time_s=time_s,
        avg_altitude_m=avg_altitude_m,
    )
    result = estimate_power_components(rider, bike, climb, headwind_m_s=headwind_m_s)
    return {
        "case": name,
        "speed_kmh": distance_m / time_s * 3.6,
        "gradient_percent": elevation_gain_m / distance_m * 100.0,
        **result,
    }


synthetic_cases = [
    synthetic_case("flat_no_wind_36_kmh", distance_m=10_000.0, elevation_gain_m=0.0, time_s=1_000.0),
    synthetic_case("flat_headwind_36_kmh", distance_m=10_000.0, elevation_gain_m=0.0, time_s=1_000.0, headwind_m_s=21/3.6),
    synthetic_case("flat_tailwind_36_kmh", distance_m=10_000.0, elevation_gain_m=0.0, time_s=1_000.0, headwind_m_s=-21/3.6),
    synthetic_case("steep_climb_20_kmh", distance_m=11_000.0, elevation_gain_m=1_000.0, time_s=1_940.0),
    synthetic_case("altitude_2000m", distance_m=10_000.0, elevation_gain_m=0.0, time_s=1_000.0, avg_altitude_m=2_000.0),
    synthetic_case("shallow_climb_24_kmh", distance_m=10_000.0, elevation_gain_m=400.0, time_s=1_200.0),
]

synthetic_df = pd.DataFrame(synthetic_cases)
synthetic_df[
    [
        "case",
        "speed_kmh",
        "gradient_percent",
        "headwind_m_s",
        "air_density_kg_m3",
        "power_gravity_w",
        "power_rolling_w",
        "power_aero_w",
        "total_power_w",
    ]
].round(2)

,case,speed_kmh,gradient_percent,headwind_m_s,air_density_kg_m3,power_gravity_w,power_rolling_w,power_aero_w,total_power_w
0,flat_no_wind_36_kmh,36.00,0.00,0.00,1.23,0.00,30.61,196.00,226.61
1,flat_headwind_36_kmh,36.00,0.00,5.83,1.23,0.00,30.61,777.99,808.60
2,flat_tailwind_36_kmh,36.00,0.00,-5.83,1.23,0.00,30.61,14.18,44.79
3,steep_climb_20_kmh,20.41,9.09,0.00,1.23,394.42,17.35,35.73,447.51
4,altitude_2000m,36.00,0.00,0.00,0.97,0.00,30.61,154.91,185.51
5,shallow_climb_24_kmh,30.00,4.00,0.00,1.23,255.06,25.51,113.43,393.99


Expected patterns:

- On a flat road, gravity power should be zero.
- At high speed, aero power should dominate rolling power.
- A headwind should increase aero power strongly because aero scales with air speed cubed.
- A tailwind should reduce aero power strongly.
- At altitude, aero power should drop because air density is lower.
- On a steep climb, gravity power should dominate.

## 2. Real-effort validation set

Fill this table with efforts where you trust the measured average power. Prefer steady climbs or time-trial-like segments with limited braking/coasting.

`drivetrain_efficiency` converts modeled resistive power at the road into estimated crank/pedal power. A typical first value is `0.97`.

In [4]:
validation_efforts = [
    {
        "name": "la_redoute_example",
        "gpx_path": REPO_ROOT / "data" / "la_redoute.gpx",
        "measured_power_w": None,  # Replace with power-meter average for this effort.
        "rider_mass_kg": 66.0,
        "bike_mass_kg": 8.0,
        "cda_m2": 0.37,
        "crr": 0.004,
        "drivetrain_efficiency": 0.97,
        "include_weather": False,
        "wind_exposure_factor": 1.0,
        "segments": None,
    },
    # Add more efforts here:
    # {
    #     "name": "my_climb",
    #     "gpx_path": REPO_ROOT / "data" / "my_climb.gpx",
    #     "measured_power_w": 315.0,
    #     "rider_mass_kg": 70.0,
    #     "bike_mass_kg": 8.0,
    #     "cda_m2": 0.37,
    #     "crr": 0.004,
    #     "drivetrain_efficiency": 0.97,
    #     "include_weather": False,
    #     "wind_exposure_factor": 1.0,
    #     "segments": None,
    # },
]

In [5]:
def build_segments(route_distance_m: float, segment_specs: list[dict] | None) -> list[RouteSegmentAdjustment] | None:
    if not segment_specs:
        return None

    return [
        RouteSegmentAdjustment(
            name=spec["name"],
            start_distance_m=float(spec["start_distance_m"]),
            end_distance_m=float(spec["end_distance_m"]),
            aero_multiplier=float(spec.get("aero_multiplier", 1.0)),
            rolling_resistance_coefficient=(
                None
                if spec.get("rolling_resistance_coefficient") is None
                else float(spec["rolling_resistance_coefficient"])
            ),
        )
        for spec in segment_specs
    ]


def evaluate_effort(effort: dict) -> dict:
    route = parse_gpx(effort["gpx_path"])
    rider = Rider(mass_kg=float(effort["rider_mass_kg"]))
    bike = Bike(
        mass_kg=float(effort["bike_mass_kg"]),
        drag_coefficient=1.0,
        frontal_area_m2=float(effort["cda_m2"]),
        rolling_resistance_coefficient=float(effort["crr"]),
    )
    summary = summarise_gpx_performance(
        effort["gpx_path"],
        rider,
        bike,
        include_weather=bool(effort.get("include_weather", False)),
        wind_exposure_factor=float(effort.get("wind_exposure_factor", 1.0)),
        segment_adjustments=build_segments(route.distance_m, effort.get("segments")),
    )

    drivetrain_efficiency = float(effort.get("drivetrain_efficiency", 1.0))
    modeled_resistive_power_w = float(summary["total_power_w"])
    modeled_crank_power_w = modeled_resistive_power_w / drivetrain_efficiency
    measured_power_w = effort.get("measured_power_w")
    modeled_w_per_kg = modeled_crank_power_w / float(effort["rider_mass_kg"])

    row = {
        "name": effort["name"],
        "measured_power_w": measured_power_w,
        "modeled_resistive_power_w": modeled_resistive_power_w,
        "modeled_crank_power_w": modeled_crank_power_w,
        "residual_w": None,
        "residual_percent": None,
        "modeled_w_per_kg": modeled_w_per_kg,
        "distance_m": summary["route_distance_m"],
        "ascent_m": summary["route_ascent_m"],
        "gradient_percent": summary["gradient_percent"],
        "speed_kmh": (summary["road_speed_m_per_s"] * 3.6),
        "vam_m_per_h": summary["vam_m_per_h"],
        "gravity_w": summary["power_gravity_w"],
        "rolling_w": summary["power_rolling_w"],
        "aero_w": summary["power_aero_w"],
        "air_density_kg_m3": summary["air_density_kg_m3"],
        "headwind_m_s": summary.get("headwind_m_s", 0.0),
        "rider_mass_kg": effort["rider_mass_kg"],
        "bike_mass_kg": effort["bike_mass_kg"],
        "cda_m2": effort["cda_m2"],
        "crr": effort["crr"],
        "drivetrain_efficiency": drivetrain_efficiency,
    }

    if measured_power_w is not None:
        row["residual_w"] = float(measured_power_w) - modeled_crank_power_w
        row["residual_percent"] = row["residual_w"] / float(measured_power_w) * 100.0

    return row


validation_df = pd.DataFrame(evaluate_effort(effort) for effort in validation_efforts)
validation_df.round(2)

,name,measured_power_w,modeled_resistive_power_w,modeled_crank_power_w,residual_w,residual_percent,modeled_w_per_kg,distance_m,ascent_m,gradient_percent,...,gravity_w,rolling_w,aero_w,air_density_kg_m3,headwind_m_s,rider_mass_kg,bike_mass_kg,cda_m2,crr,drivetrain_efficiency
0,la_redoute_example,None,575.64,593.45,None,None,8.99,1499.8,153.1,10.21,...,491.78,19.27,64.6,1.19,0.0,66.0,8.0,0.37,0.0,0.97


## 3. Residual diagnostics

Use the patterns below to infer which physical variables are missing or misestimated.

In [6]:
def diagnose_row(row: pd.Series) -> str:
    if pd.isna(row.get("residual_w")):
        return "Add measured_power_w to diagnose this effort."

    residual_percent = float(row["residual_percent"])
    speed_kmh = float(row["speed_kmh"])
    gradient_percent = float(row["gradient_percent"])

    if abs(residual_percent) <= 5.0:
        return "Good agreement for a first-order field model."

    if residual_percent > 0.0 and speed_kmh >= 28.0:
        return "Model is low at high speed: check CdA, headwind, drafting, and wind exposure."
    if residual_percent > 0.0 and gradient_percent >= 6.0:
        return "Model is low on a climb: check mass, elevation gain, drivetrain loss, and altitude correction."
    if residual_percent > 0.0:
        return "Model is low: check Crr, drivetrain loss, wind, and stop/start energy."
    if residual_percent < 0.0 and speed_kmh >= 28.0:
        return "Model is high at high speed: CdA may be too high or tailwind/drafting is missing."
    if residual_percent < 0.0 and gradient_percent >= 6.0:
        return "Model is high on a climb: mass or elevation gain may be too high."
    return "Model is high: check Crr, CdA, tailwind, coasting, and measured-power scope."


diagnostics_df = validation_df.copy()
diagnostics_df["diagnosis"] = diagnostics_df.apply(diagnose_row, axis=1)
diagnostics_df[["name", "measured_power_w", "modeled_crank_power_w", "residual_w", "residual_percent", "diagnosis"]].round(2)

,name,measured_power_w,modeled_crank_power_w,residual_w,residual_percent,diagnosis
0,la_redoute_example,None,593.45,None,None,Add measured_power_w to diagnose this effort.


## 4. Sensitivity sweep

Use this to see which assumptions move the answer most. Start with CdA and Crr because they are hard to know precisely without testing.

In [7]:
base_effort = validation_efforts[0]

sweep_rows = []
for cda_m2 in [0.28, 0.32, 0.37, 0.42, 0.48]:
    effort = {**base_effort, "name": f"CdA {cda_m2:.2f}", "cda_m2": cda_m2}
    sweep_rows.append(evaluate_effort(effort))

for crr in [0.0025, 0.0035, 0.0040, 0.0055, 0.0070]:
    effort = {**base_effort, "name": f"Crr {crr:.4f}", "crr": crr}
    sweep_rows.append(evaluate_effort(effort))

sweep_df = pd.DataFrame(sweep_rows)
sweep_df[["name", "modeled_crank_power_w", "modeled_w_per_kg", "gravity_w", "rolling_w", "aero_w", "speed_kmh", "gradient_percent"]].round(2)

,name,modeled_crank_power_w,modeled_w_per_kg,gravity_w,rolling_w,aero_w,speed_kmh,gradient_percent
0,CdA 0.28,577.25,8.75,491.78,19.27,48.88,23.89,10.21
1,CdA 0.32,584.45,8.86,491.78,19.27,55.87,23.89,10.21
2,CdA 0.37,593.45,8.99,491.78,19.27,64.60,23.89,10.21
3,CdA 0.42,602.45,9.13,491.78,19.27,73.33,23.89,10.21
4,CdA 0.48,613.25,9.29,491.78,19.27,83.80,23.89,10.21
5,Crr 0.0025,586.00,8.88,491.78,12.04,64.60,23.89,10.21
6,Crr 0.0035,590.96,8.95,491.78,16.86,64.60,23.89,10.21
7,Crr 0.0040,593.45,8.99,491.78,19.27,64.60,23.89,10.21
8,Crr 0.0055,600.90,9.10,491.78,26.50,64.60,23.89,10.21
9,Crr 0.0070,608.35,9.22,491.78,33.72,64.60,23.89,10.21


## 5. Optional segment modeling

To validate protected vs solo sections, add `segments` to an effort. Distances are from the route start in meters.

Example:

```python
"segments": [
    {"name": "protected", "start_distance_m": 0.0, "end_distance_m": 900.0, "aero_multiplier": 0.65},
    {"name": "solo", "start_distance_m": 900.0, "end_distance_m": 1500.0, "aero_multiplier": 1.0},
]
```

If measured power diverges mostly when segment protection changes, the likely missing variable is not the base physics equation but the aero multiplier or where the segment boundaries are placed.